In [ ]:

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# -----------------------------
# 1. PROBLEM DEFINITION
# -----------------------------
SUBJECTS = [
    "Mathematics", "Deep Learning", "Computer Networks",
    "Database Systems", "Operating Systems", "Artificial Intelligence"
]

SLOTS = ["Slot 1", "Slot 2", "Slot 3"]

# Subjects sharing students must not have the same exam slot.
CONFLICT_PAIRS = [
    (0, 1), (0, 2), (1, 4),
    (2, 3), (3, 5), (4, 5)
]

def clashes(chromosome):
    return sum(chromosome[a] == chromosome[b]
               for a, b in CONFLICT_PAIRS)

def fitness(chromosome):
    return 1 / (1 + clashes(chromosome))

def create_population(size):
    return [
        [random.randrange(len(SLOTS)) for _ in SUBJECTS]
        for _ in range(size)
    ]

# -----------------------------
# 2. SELECTION
# -----------------------------
def select_parent(population, method):
    if method == "Roulette-wheel":
        scores = np.array([fitness(x) for x in population])
        probabilities = scores / scores.sum()
        return population[
            np.random.choice(len(population), p=probabilities)
        ][:]

    # Tournament selection
    contestants = random.sample(
        population, min(3, len(population))
    )
    return max(contestants, key=fitness)[:]

# -----------------------------
# 3. CROSSOVER
# -----------------------------
def crossover(p1, p2, probability):
    if len(p1) > 1 and random.random() < probability:
        point = random.randint(1, len(p1) - 1)
        c1 = p1[:point] + p2[point:]
        c2 = p2[:point] + p1[point:]
        return c1, c2, point

    return p1[:], p2[:], None

# -----------------------------
# 4. MUTATION
# -----------------------------
def mutate(chromosome, probability):
    child = chromosome[:]
    changed = []

    for i in range(len(child)):
        if random.random() < probability:
            old_slot = child[i]
            possible_slots = [
                s for s in range(len(SLOTS))
                if s != old_slot
            ]
            child[i] = random.choice(possible_slots)
            changed.append(i)

    return child, changed

# -----------------------------
# 5. RUN GENETIC ALGORITHM
# -----------------------------
def run_ga(pop_size, generations, selection_method,
           crossover_rate, mutation_rate):

    population = create_population(pop_size)
    history = []
    operator_examples = []

    for generation in range(generations + 1):
        scores = [fitness(x) for x in population]

        history.append({
            "Generation": generation,
            "Best fitness": max(scores),
            "Average fitness": np.mean(scores),
            "Fewest clashes": min(clashes(x) for x in population)
        })

        if generation == generations:
            break

        new_population = [
            max(population, key=fitness)[:]
        ]  # Elitism: preserve the best solution

        while len(new_population) < pop_size:
            p1 = select_parent(population, selection_method)
            p2 = select_parent(population, selection_method)

            c1, c2, point = crossover(
                p1, p2, crossover_rate
            )

            c1, m1 = mutate(c1, mutation_rate)
            c2, m2 = mutate(c2, mutation_rate)

            if len(operator_examples) < 3:
                operator_examples.append({
                    "Parent 1": p1,
                    "Parent 2": p2,
                    "Crossover point": point,
                    "Child 1": c1,
                    "Child 2": c2,
                    "Mutation genes in child 1": m1,
                    "Mutation genes in child 2": m2
                })

            new_population.append(c1)
            if len(new_population) < pop_size:
                new_population.append(c2)

        population = new_population

    return population, pd.DataFrame(history), operator_examples

# -----------------------------
# 6. INTERACTIVE CONTROLS
# -----------------------------
pop_widget = widgets.IntSlider(
    value=12, min=6, max=40, step=2,
    description="Population"
)

gen_widget = widgets.IntSlider(
    value=30, min=1, max=100, step=1,
    description="Generations"
)

selection_widget = widgets.Dropdown(
    options=["Tournament", "Roulette-wheel"],
    value="Tournament",
    description="Selection"
)

crossover_widget = widgets.FloatSlider(
    value=0.8, min=0, max=1, step=0.05,
    description="Crossover"
)

mutation_widget = widgets.FloatSlider(
    value=0.08, min=0, max=0.5, step=0.01,
    description="Mutation"
)

run_button = widgets.Button(
    description="Run Genetic Algorithm",
    button_style="success",
    icon="play"
)

output = widgets.Output()

def run_demo(_):
    with output:
        clear_output(wait=True)

        random.seed(42)
        np.random.seed(42)

        population, history, examples = run_ga(
            pop_widget.value,
            gen_widget.value,
            selection_widget.value,
            crossover_widget.value,
            mutation_widget.value
        )

        best = max(population, key=fitness)

        print("BEST TIMETABLE")
        print("Chromosome:", best)
        print("Fitness:", round(fitness(best), 3))
        print("Number of clashes:", clashes(best))

        timetable = pd.DataFrame({
            "Subject": SUBJECTS,
            "Exam slot": [SLOTS[x] for x in best]
        })
        display(timetable)

        print("\nFINAL POPULATION")
        table = pd.DataFrame([
            {
                "Candidate": i + 1,
                "Chromosome": str(ch),
                "Clashes": clashes(ch),
                "Fitness": round(fitness(ch), 3)
            }
            for i, ch in enumerate(population)
        ]).sort_values("Clashes")
        display(table)

        plt.figure(figsize=(9, 4))
        plt.plot(
            history["Generation"],
            history["Best fitness"],
            label="Best fitness",
            marker="."
        )
        plt.plot(
            history["Generation"],
            history["Average fitness"],
            label="Average fitness"
        )
        plt.xlabel("Generation")
        plt.ylabel("Fitness")
        plt.title("Genetic Algorithm: Fitness Progress")
        plt.legend()
        plt.grid(True)
        plt.show()

        print("\nSELECTION → CROSSOVER → MUTATION EXAMPLES")
        for i, example in enumerate(examples, 1):
            print(f"\nExample {i}")
            for key, value in example.items():
                print(f"{key}: {value}")

        if clashes(best) == 0:
            print("\nA timetable satisfying all modeled conflict constraints was found.")
        else:
            print("\nSome conflicts remain. Try changing the settings.")

run_button.on_click(run_demo)

display(widgets.VBox([
    pop_widget,
    gen_widget,
    selection_widget,
    crossover_widget,
    mutation_widget,
    run_button,
    output
]))


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Conv2DTranspose,
    concatenate
)

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


def jaccard_index(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        - intersection
    )

    return (intersection + smooth) / (union + smooth)


In [ ]:
import tensorflow as tf
import numpy as np

# Dataset configuration
IMG_SIZE = (128, 128)
NUM_TRAIN_IMAGES = 500
NUM_TEST_IMAGES = 100
BATCH_SIZE = 4

print(f"Training dataset: {NUM_TRAIN_IMAGES} images (Synthetic)")
print(f"Testing dataset: {NUM_TEST_IMAGES} images (Synthetic)")

# Generate synthetic image and segmentation mask tensors
def generate_synthetic_data(num_images):
    # Random images normalized between 0.0 and 1.0
    images = np.random.rand(num_images, IMG_SIZE[0], IMG_SIZE[1], 3).astype(np.float32)
    # Binary masks (0 or 1)
    masks = np.random.randint(0, 2, size=(num_images, IMG_SIZE[0], IMG_SIZE[1], 1)).astype(np.float32)
    return images, masks

# Create data arrays
train_images, train_masks = generate_synthetic_data(NUM_TRAIN_IMAGES)
test_images, test_masks = generate_synthetic_data(NUM_TEST_IMAGES)

# Create training dataset pipeline
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_images, train_masks))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Create testing dataset pipeline
test_ds = (
    tf.data.Dataset.from_tensor_slices((test_images, test_masks))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training and testing datasets are ready!")

Training dataset: 500 images (Synthetic)
Testing dataset: 100 images (Synthetic)
Training and testing datasets are ready!


In [ ]:
def create_segmentation_generator(image_dir, mask_dir, batch_size=4):
    """
    Creates a generator that yields paired image and mask batches.
    Adapted for MedPlant dataset where images and masks are in separate directories.
    """

    # Get sorted list of image files
    image_files = sorted(
        glob.glob(os.path.join(image_dir, "*.jpg")) +
        glob.glob(os.path.join(image_dir, "*.png"))
    )

    if not image_files:
        raise ValueError(f"No images found in {image_dir}")

    # Verify corresponding masks exist
    mask_files = []
    for img_file in image_files:
        base_name = os.path.basename(img_file)
        mask_file = os.path.join(mask_dir, base_name)
        if os.path.exists(mask_file):
            mask_files.append(mask_file)
        else:
            print(f"Warning: No mask found for {img_file}")

    print(f"Found {len(image_files)} images and {len(mask_files)} masks")

    if len(mask_files) == 0:
        raise ValueError(f"No matching masks found in {mask_dir}")

    # Generator function
    def generator():
        idx = 0
        while True:
            batch_images = []
            batch_masks = []
            for _ in range(batch_size):
                if idx >= len(image_files):
                    idx = 0  # Loop back to start

                # Load image
                img = tf.keras.preprocessing.image.load_img(
                    image_files[idx],
                    target_size=(128, 128),
                    color_mode='grayscale'
                )

                img_array = tf.keras.preprocessing.image.img_to_array(img)
                img_array = img_array / 255.0  # Normalize
                batch_images.append(img_array)                # Load corresponding mask
                if idx < len(mask_files):
                    mask = tf.keras.preprocessing.image.load_img(
                        mask_files[idx],
                        target_size=(128, 128),
                        color_mode='grayscale'
                    )

                    mask_array = tf.keras.preprocessing.image.img_to_array(mask)
                    mask_array = mask_array / 255.0  # Normalize
                    batch_masks.append(mask_array)

                idx += 1

            yield (np.array(batch_images), np.array(batch_masks))

    return generator()

In [ ]:
batch_size = 4

# Since we are using the Oxford-IIIT Pet dataset from tf.data,
# we can assign train_ds and test_ds directly to our generator variables.
train_gen = train_ds
test_gen = test_ds

In [ ]:
def build_unet(input_shape=(128, 128, 3)):
    inputs = Input(input_shape)

    c1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = MaxPooling2D(2)(c1)

    c2 = Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = MaxPooling2D(2)(c2)

    c3 = Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = Conv2D(256, 3, activation='relu', padding='same')(c3)
    p3 = MaxPooling2D(2)(c3)

    c4 = Conv2D(512, 3, activation='relu', padding='same')(p3)
    c4 = Conv2D(512, 3, activation='relu', padding='same')(c4)
    p4 = MaxPooling2D(2)(c4)

    c5 = Conv2D(1024, 3, activation='relu', padding='same')(p4)
    c5 = Conv2D(1024, 3, activation='relu', padding='same')(c5)

    u6 = Conv2DTranspose(512, 2, strides=2, padding='same')(c5)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(512, 3, activation='relu', padding='same')(u6)
    c6 = Conv2D(512, 3, activation='relu', padding='same')(c6)
    u6 = Conv2DTranspose(512, 2, strides=2, padding='same')(c5)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(512, 3, activation='relu', padding='same')(u6)
    c6 = Conv2D(512, 3, activation='relu', padding='same')(c6)

    u7 = Conv2DTranspose(256, 2, strides=2, padding='same')(c6)
    u7 = concatenate([u7, c3])
    c7 = Conv2D(256, 3, activation='relu', padding='same')(u7)
    c7 = Conv2D(256, 3, activation='relu', padding='same')(c7)

    u8 = Conv2DTranspose(128, 2, strides=2, padding='same')(c7)
    u8 = concatenate([u8, c2])
    c8 = Conv2D(128, 3, activation='relu', padding='same')(u8)
    c8 = Conv2D(128, 3, activation='relu', padding='same')(c8)
    u9 = Conv2DTranspose(64, 2, strides=2, padding='same')(c8)
    u9 = concatenate([u9, c1])
    c9 = Conv2D(64, 3, activation='relu', padding='same')(u9)
    c9 = Conv2D(64, 3, activation='relu', padding='same')(c9)

    outputs = Conv2D(1, 1, activation='sigmoid')(c9)

    model = Model(inputs, outputs)
    return model